# Warehouse DreamerV3 — Offline Training on Colab A100

Isaac Sim does **not** run on Colab (no Omniverse, fixed driver, no offscreen camera).
So compute is split:

1. **Collect locally** (your RTX 5050, sim running):
   ```
   python scripts/collect_offline.py --steps 20000 --out training/data/episodes --headless
   ```
   Then zip `training/data/episodes/` and put it on Google Drive.
2. **Train here** (A100): load the episodes, train the world model + actor-critic
   offline. No sim, no env.

Runtime > Change runtime type > **A100 GPU** before running.

> Offline limit: a random-policy dataset teaches the world model dynamics well, but the
> actor can only get as good as the states the data visited. For a stronger policy,
> collect with a scripted approach policy (see the `ponytail:` note in collect_offline.py).

In [ ]:
# Confirm the GPU is an A100
!nvidia-smi

## 1. Get the code

Public repo: clone directly. Private repo: use a token
`https://<TOKEN>@github.com/henray404/BANTAI_FP.git`, or skip cloning and copy the
repo folder from Drive instead.

In [ ]:
%cd /content
![ -d BANTAI_FP ] || git clone https://github.com/henray404/BANTAI_FP.git
%cd /content/BANTAI_FP

In [ ]:
# Minimal deps. Colab already has torch+CUDA / numpy / tensorboard.
!pip install -q -r requirements-colab.txt

## 2. Point to the dataset

Mount Drive and set `DATA` to the folder of `.npz` episodes you collected locally.
If you uploaded a zip, unzip it first (next cell).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT this to where your episodes live on Drive:
DATA = '/content/drive/MyDrive/BANTAI_FP/episodes'

# If you uploaded episodes.zip instead, uncomment:
# !mkdir -p /content/episodes && unzip -q '/content/drive/MyDrive/BANTAI_FP/episodes.zip' -d /content/episodes
# DATA = '/content/episodes'

import os
print('episodes found:', len([f for f in os.listdir(DATA) if f.endswith('.npz')]))

## 3. Train

Sanity-check the script first (no data/GPU needed), then run the real thing.
`--batch_size 32` suits the A100; raise `--steps` for a longer run. Re-running resumes
from `latest.pt`.

In [ ]:
!python scripts/train_offline.py --self_check

In [ ]:
!python scripts/train_offline.py \
  --data "$DATA" \
  --steps 100000 \
  --batch_size 32 \
  --device cuda \
  --logdir training/results/offline

## 4. Watch metrics + grab the checkpoint

In [ ]:
%load_ext tensorboard
%tensorboard --logdir training/results/offline

In [ ]:
# Save the checkpoint back to Drive (survives the session)
!cp training/results/offline/latest.pt '/content/drive/MyDrive/BANTAI_FP/latest.pt'
print('saved latest.pt to Drive')